In [11]:
# Run this in a cell before exporting
%pip install onnx onnxsim onnxruntime-gpu

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from ultralytics import YOLO
import yaml

DATASET_PATH = "augmentation_dataset"

In [3]:
def create_dataset_yaml(dataset_path="augmentation_dataset", num_classes=1, class_names=None):
    """Create dataset YAML configuration with proper formatting"""
    if class_names is None:
        class_names = {0: 'pest'}
    
    yaml_content = {
        'path': os.path.abspath(dataset_path),
        'train': 'train/images',
        'val': 'valid/images', 
        'test': 'test/images',
        'nc': num_classes,
        'names': class_names
    }
    
    # Save YAML
    with open("data.yaml", "w") as f:
        yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)
    
    print("✅ data.yaml created!")
    print(f"Dataset path: {os.path.abspath(dataset_path)}")
    return "data.yaml"

# Create dataset configuration
yaml_path = create_dataset_yaml(DATASET_PATH)

✅ data.yaml created!
Dataset path: c:\Users\plebj\Desktop\School\CS5814\augmentation_dataset


In [4]:
def verify_dataset_structure(dataset_path):
    """Verify dataset has correct structure"""
    required_dirs = [
        "train/images", "train/labels",
        "valid/images", "valid/labels", 
        "test/images", "test/labels"
    ]
    
    print("🔍 Verifying dataset structure...")
    for dir_path in required_dirs:
        full_path = os.path.join(dataset_path, dir_path)
        if not os.path.exists(full_path):
            print(f"❌ Missing: {full_path}")
            return False
        # Check if directory has files
        files = os.listdir(full_path)
        if not files:
            print(f"⚠️  Empty directory: {full_path}")
        else:
            print(f"✅ {dir_path}: {len(files)} files")
    
    return True

# Verify dataset
dataset_valid = verify_dataset_structure(DATASET_PATH)
if not dataset_valid:
    print("❌ Dataset structure invalid! Fix before training.")

🔍 Verifying dataset structure...
✅ train/images: 4957 files
✅ train/labels: 4957 files
✅ valid/images: 356 files
✅ valid/labels: 356 files
✅ test/images: 178 files
✅ test/labels: 178 files


In [5]:
def train_model(yaml_path="data.yaml"):
    """Main training function with enhanced configuration"""

    # Load YOLO11 model
    print("🚀 Loading YOLO11n model...")
    model = YOLO("yolo11n.pt")
    
    # Enhanced training configuration
    results = model.train(
        data=yaml_path,
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,
        patience=10,           # Early stopping patience
        lr0=0.01,              # Initial learning rate
        lrf=0.01,              # Final learning rate factor
        momentum=0.937,         # SGD momentum
        weight_decay=0.0005,    # Optimizer weight decay
        warmup_epochs=3.0,      # Warmup epochs
        warmup_momentum=0.8,    # Warmup initial momentum
        box=7.5,               # Box loss gain
        cls=0.5,               # Classification loss gain
        hsv_h=0.015,           # Image HSV-Hue augmentation
        hsv_s=0.7,             # Image HSV-Saturation augmentation
        hsv_v=0.4,             # Image HSV-Value augmentation
        degrees=0.0,           # Image rotation
        translate=0.1,         # Image translation
        scale=0.5,             # Image scale
        shear=0.0,             # Image shear
        perspective=0.0,       # Image perspective
        flipud=0.0,            # Image flip up-down
        fliplr=0.5,            # Image flip left-right
        mosaic=1.0,            # Image mosaic
        mixup=0.0,             # Image mixup
        copy_paste=0.0,        # Segment copy-paste
        save=True,             # Save train checkpoints
        save_period=-1,        # Save checkpoint every x epochs
        cache=False,           # Cache images in RAM/disk
        workers=8,             # Dataloader workers
        single_cls=True,       # Train as single-class dataset
        optimizer='auto',      # Optimizer selection
        verbose=True,          # Print results per epoch
        seed=42,               # Training seed
        deterministic=True,    # Use deterministic algorithms
        plots=True             # Save plots during training
    )
    
    return model, results

# Train the model (this will take ~25 minutes)
model, results = train_model(yaml_path)

🚀 Loading YOLO11n model...
Ultralytics 8.3.204  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspective=0.0, 

In [6]:
def evaluate_model(model, yaml_path="data.yaml"):
    """Comprehensive model evaluation"""
    print("📊 Evaluating model...")
    
    # Validation metrics
    metrics = model.val(
        data=yaml_path,
        split="val",
        save_json=True,    # Save results to JSON
        save_hybrid=True,  # Save hybrid version of labels
        conf=0.5,          # Object confidence threshold
        iou=0.6,           # NMS IoU threshold
        plots=True         # Save plots
    )
    
    # Test set evaluation  
    test_metrics = model.val(
        data=yaml_path,
        split="test",
        conf=0.5,
        iou=0.6
    )
    
    return metrics, test_metrics

# Evaluate the trained model
val_metrics, test_metrics = evaluate_model(model)

📊 Evaluating model...
WARNING 'save_hybrid' is deprecated and will be removed in the future.
Ultralytics 8.3.204  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 938.9607.0 MB/s, size: 43.1 KB)
val: Scanning C:\Users\plebj\Desktop\School\CS5814\augmentation_dataset\valid\labels.cache... 356 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 356/356 232.2Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 13.8it/s 1.7s0.1s
                   all        356        365      0.997      0.995      0.995      0.926
Speed: 0.3ms preprocess, 2.2ms inference, 0.0ms loss, 0.4ms postprocess per image
Saving C:\Users\plebj\Desktop\School\CS5814\runs\detect\train2\predictions.json...
Results saved to C:\Users\plebj\Desktop\School\CS5814\runs\detect\train2
Ultralytics 8.3

In [12]:
def export_model(model, export_formats=["onnx", "tflite"]):
    """Export model to various formats"""
    print("📤 Exporting model...")
    
    exported_paths = {}
    
    for format in export_formats:
        try:
            if format == "tflite":
                # TFLite export for Raspberry Pi
                path = model.export(format=format, int8=True, data="data.yaml")
            else:
                path = model.export(format=format)
            
            exported_paths[format] = path
            print(f"✅ Exported to {format}: {path}")
            
        except Exception as e:
            print(f"❌ Failed to export {format}: {e}")
    
    return exported_paths

# Export the model
exported_paths = export_model(model, export_formats=["onnx"])  # "tflite" if needed

📤 Exporting model...
Ultralytics 8.3.204  Python-3.11.9 torch-2.5.1+cu121 CPU (AMD Ryzen 7 9700X 8-Core Processor)

PyTorch: starting from 'C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.2 MB)

ONNX: starting export with onnx 1.19.0 opset 19...
ONNX: slimming with onnxslim 0.1.70...
ONNX: export success  1.0s, saved as 'C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\weights\best.onnx' (10.1 MB)

Export complete (1.1s)
Results saved to C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\weights
Predict:         yolo predict task=detect model=C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\weights\best.onnx imgsz=640  
Validate:        yolo val task=detect model=C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\weights\best.onnx imgsz=640 data=data.yaml  
Visualize:       https://netron.app
✅ Exported to onnx: C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\we

In [22]:
def test_inference(model, dataset_path="augmentation_dataset"):
    """Test model inference on sample images"""
    print("🧪 Testing inference...")
    
    test_dir = os.path.join(dataset_path, "test/images")
    if os.path.exists(test_dir):
        test_images = [f for f in os.listdir(test_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        if test_images:
            # Test on first 3 images
            for i, img_name in enumerate(test_images[:3]):
                img_path = os.path.join(test_dir, img_name)
                print(f"Testing: {img_name}")
                
                results = model(img_path, conf=0.90)  # Confidence threshold
                
                # Show results (optional - comment out for headless environments)
                # results[0].show()
                
                # Print detection info
                for r in results:
                    if len(r.boxes) > 0:
                        print(f"  Detected {len(r.boxes)} pests")
                    else:
                        print("  No pests detected")
            
            print("✅ Inference test completed!")
        else:
            print("⚠️  No test images found")
    else:
        print("⚠️  Test directory not found")

# Test inference
test_inference(model, DATASET_PATH)

🧪 Testing inference...
Testing: 01264570-7778-4f5b-aaaf-3a5dcf963721.jpg

image 1/1 c:\Users\plebj\Desktop\School\CS5814\augmentation_dataset\test\images\01264570-7778-4f5b-aaaf-3a5dcf963721.jpg: 640x640 1 item, 38.1ms
Speed: 2.3ms preprocess, 38.1ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 640)
  Detected 1 pests
Testing: 05cca704-2878-4c66-bb7b-0b8da1afe7c5.jpg

image 1/1 c:\Users\plebj\Desktop\School\CS5814\augmentation_dataset\test\images\05cca704-2878-4c66-bb7b-0b8da1afe7c5.jpg: 640x640 1 item, 42.3ms
Speed: 2.7ms preprocess, 42.3ms inference, 5.0ms postprocess per image at shape (1, 3, 640, 640)
  Detected 1 pests
Testing: 09328ed9-65b8-4fe4-a24e-7357089f1a73.jpg

image 1/1 c:\Users\plebj\Desktop\School\CS5814\augmentation_dataset\test\images\09328ed9-65b8-4fe4-a24e-7357089f1a73.jpg: 640x640 1 item, 69.1ms
Speed: 6.1ms preprocess, 69.1ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 640)
  Detected 1 pests
✅ Inference test completed!


In [ ]:
print("\n🎉 Training completed successfully!")
print(f"📁 Model saved in: {os.path.join('runs', 'detect', 'train')}")

# Print final metrics 
if hasattr(results, 'results_dict'):
    print(f"📊 Final mAP@0.5: {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")

# Fix for test metrics - they're stored differently
if test_metrics is not None:
    # Try different attribute names that Ultralytics uses
    if hasattr(test_metrics, 'box_map'):
        print(f"📊 Test mAP@0.5: {test_metrics.box_map}")
    elif hasattr(test_metrics, 'map50'):
        print(f"📊 Test mAP@0.5: {test_metrics.map50}")
    elif hasattr(test_metrics, 'results_dict'):
        print(f"📊 Test mAP@0.5: {test_metrics.results_dict.get('metrics/mAP50(B)', 'N/A')}")
    else:
        print(f"📊 Test mAP@0.5: {getattr(test_metrics, 'box_map50', 'N/A')}")
else:
    print("📊 Test mAP@0.5: N/A (test_metrics is None)")


🎉 Training completed successfully!
📁 Model saved in: runs\detect\train
📊 Final mAP@0.5: 0.995
📊 Test mAP@0.5: 0.995
